In [3]:
import pandas as pd
import numpy as np

In [4]:
train = pd.read_csv('data/train.csv')

test = pd.read_csv('data/test.csv')

In [5]:
cols_fillna = ['PoolQC','MiscFeature','Alley','Fence','MasVnrType','FireplaceQu',
               'GarageQual','GarageCond','GarageFinish','GarageType', 'Electrical',
               'KitchenQual', 'SaleType', 'Functional', 'Exterior2nd', 'Exterior1st',
               'BsmtExposure','BsmtCond','BsmtQual','BsmtFinType1','BsmtFinType2',
               'MSZoning', 'Utilities']

for col in cols_fillna:
    train[col].fillna('None',inplace=True)
    test[col].fillna('None',inplace=True)


train_means = train.mean(numeric_only=True)
test_means = test.mean(numeric_only=True)

train.fillna(train_means, inplace=True)
test.fillna(test_means, inplace=True)

train = train[train.GrLivArea < 4500]
train.reset_index(drop=True, inplace=True)
train["SalePrice"] = np.log1p(train["SalePrice"])

y = train['SalePrice'].reset_index(drop=True)

train_features = train.drop(['SalePrice'], axis=1)
test_features = test
features = pd.concat([train_features, test_features]).reset_index(drop=True)

all_features = pd.concat([train_features, test_features]).reset_index(drop=True)

features = features.drop(['Utilities', 'Street', 'PoolQC',], axis=1)

features['YrBltAndRemod']=features['YearBuilt']+features['YearRemodAdd']
features['TotalSF']=features['TotalBsmtSF'] + features['1stFlrSF'] + features['2ndFlrSF']

features['Total_sqr_footage'] = (features['BsmtFinSF1'] + features['BsmtFinSF2'] +
                                 features['1stFlrSF'] + features['2ndFlrSF'])

features['Total_Bathrooms'] = (features['FullBath'] + (0.5 * features['HalfBath']) +
                               features['BsmtFullBath'] + (0.5 * features['BsmtHalfBath']))

features['Total_porch_sf'] = (features['OpenPorchSF'] + features['3SsnPorch'] +
                              features['EnclosedPorch'] + features['ScreenPorch'] +
                              features['WoodDeckSF'])


features['haspool'] = features['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
features['has2ndfloor'] = features['2ndFlrSF'].apply(lambda x: 1 if x > 0 else 0)
features['hasgarage'] = features['GarageArea'].apply(lambda x: 1 if x > 0 else 0)
features['hasbsmt'] = features['TotalBsmtSF'].apply(lambda x: 1 if x > 0 else 0)
features['hasfireplace'] = features['Fireplaces'].apply(lambda x: 1 if x > 0 else 0)

final_features = pd.get_dummies(features).reset_index(drop=True)


/var/folders/0x/33gpdf5s7295nplw560g6d9w0000gn/T/ipykernel_72885/482553963.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna('None',inplace=True)
/var/folders/0x/33gpdf5s7295nplw560g6d9w0000gn/T/ipykernel_72885/482553963.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves a

In [ ]:
X_train_raw = train.drop(columns=["SalePrice"])
X_all = pd.concat([X_train_raw, test], axis=0)

X_all = pd.get_dummies(X_all)

X = X_all.iloc[:len(train), :].copy()
X_test = X_all.iloc[len(train):, :].copy()

In [7]:
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.linear_model import Ridge, LinearRegression, RidgeCV,Lasso, ElasticNet
from sklearn.ensemble import StackingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from catboost import CatBoostRegressor

In [ ]:
estimators = [
    ('cb', CatBoostRegressor(loss_function="RMSE",
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3,
    subsample=0.8,
    colsample_bylevel=0.8,
    random_seed=42,
    verbose=False)),
    ('r', Ridge(alpha=20, solver='svd')),
    ('gb',GradientBoostingRegressor(random_state=42))
]

final_estimator = RidgeCV(alphas=[0.1, 1, 10, 100])
kf = KFold(n_splits=5, shuffle=True, random_state=42)

model = StackingRegressor(
    estimators=estimators,
    final_estimator=final_estimator,
    n_jobs=-1,
    cv=kf
)
model.fit(X, y)

pred_log = model.predict(X_test)

pred = np.expm1(pred_log)

submission = pd.DataFrame({
    "Id": test["Id"],          
    "SalePrice": np.round(pred,7)
})

submission.to_csv("submission.csv", index=False)
print(submission.head())


scores = cross_val_score(model, X, y, cv=kf, scoring="neg_root_mean_squared_error")
rmse = -scores

print("RMSE по фолдам:", rmse)
print("CV mean:", rmse.mean())
print("CV std:", rmse.std())



     Id      SalePrice
0  1461  120959.607525
1  1462  155965.801438
2  1463  183967.718382
3  1464  196329.700131
4  1465  186076.271041
RMSE по фолдам: [0.11570754 0.10301903 0.11675559 0.11658534 0.09569167]
CV mean: 0.10955183303838026
CV std: 0.008649146998596367
